# Sharadar Examples

Demonstrates fetching financial data using the Sharadar (Nasdaq Data Link) provider.

**Prerequisites**:
- `pip install vertex-forager`
- A valid `SHARADAR_API_KEY` environment variable

## 1. Setup

In [1]:
import os

api_key = os.environ.get("SHARADAR_API_KEY")
if not api_key:
    raise RuntimeError(
        "Set the SHARADAR_API_KEY environment variable before running this notebook.\n"
        "  export SHARADAR_API_KEY=your_key_here"
    )

## 2. Fetch Price Data (In-Memory)

In [3]:
from vertex_forager import create_client

client = create_client(
    provider="sharadar",
    api_key=api_key,
    rate_limit=500,
)

df = client.get_price_data(tickers=["AAPL", "MSFT"], show_progress=False)
df.head()

HTML(value='⏳ Prefetching metadata for smart batching...')

Fetching price data:   0%|          | 0/2 [00:00<?, ?tickers/s]

HTML(value='⏳ Finalizing database writes...')

provider,ticker,date,open,high,low,close,volume,closeadj,closeunadj,lastupdated,fetched_at
str,str,date,f64,f64,f64,f64,i64,f64,f64,date,"datetime[μs, UTC]"
"""sharadar""","""AAPL""",1997-12-31,0.117,0.122,0.116,0.117,406358000,0.098,13.13,2026-02-10,2026-03-21 08:59:48.112728 UTC
"""sharadar""","""AAPL""",1998-01-02,0.122,0.145,0.12,0.145,718110000,0.122,16.25,2026-02-10,2026-03-21 08:59:48.112728 UTC
"""sharadar""","""AAPL""",1998-01-05,0.147,0.148,0.136,0.142,651874000,0.119,15.88,2026-02-10,2026-03-21 08:59:48.112728 UTC
"""sharadar""","""AAPL""",1998-01-06,0.142,0.178,0.132,0.169,1812474000,0.142,18.94,2026-02-10,2026-03-21 08:59:48.112728 UTC
"""sharadar""","""AAPL""",1998-01-07,0.168,0.17,0.154,0.156,1041622000,0.131,17.5,2026-02-10,2026-03-21 08:59:48.112728 UTC


## 3. Persist to DuckDB

Pass `connect_db` with a `duckdb://` URI to write results to a local DuckDB file.

In [ ]:
res = client.get_price_data(
    tickers=["AAPL", "MSFT", "GOOGL"],
    connect_db="duckdb://./sharadar_demo.duckdb",
    show_progress=False,
)
print(res)

## 4. Multiple Datasets

Sharadar supports: `price`, `daily`, `fundamental`, `actions`, `tickers`, `sp500`.

In [ ]:
# Fetch fundamental data
fundamentals = client.get_data(
    tickers=["AAPL"],
    datasets=["fundamental"],
    show_progress=False,
)
fundamentals.head()

## 5. Custom Configuration

Adjust rate limits, concurrency, and enable metrics for the Sharadar provider.

In [ ]:
custom_client = create_client(
    provider="sharadar",
    api_key=api_key,
    rate_limit=60,
    concurrency=2,
    metrics_enabled=True,
)

res = custom_client.get_price_data(
    tickers=["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"],
    connect_db="duckdb://./sharadar_custom.duckdb",
    show_progress=False,
)
print(res)

## Cleanup

In [ ]:
from pathlib import Path

for f in Path(".").glob("sharadar_*.duckdb*"):
    f.unlink(missing_ok=True)
    print(f"Removed {f}")